# SOPQuery — RAG Pipeline
## Sprint 1 - Stage 1: PDF Ingestion
### Functions
- Successfully load PDFs without error.
- Verify content of extracted text that is readible, no error present and non-empty.


In [1]:
# import libraries
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

/var/folders/my/5yhvxszd4rz8fvy2bm61s3_80000gn/T/ipykernel_35947/1752622643.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
# load pdfs saved in data/FDA folder
# save all loaded documents into a variable: documents

pdf_path = Path("../data/FDA")

documents = []
for pdf in pdf_path.glob('*.pdf'):
    try:
        pdf_loader = PyPDFLoader(pdf)
        documents.extend(pdf_loader.load())
    
    # validate successful PDF ingestion
    except Exception as err:
        print(f'Failed to load {pdf.name}: {err}')

# check output to verify all 5 pdfs successfully loaded
expected = len(list(pdf_path.glob('*.pdf')))
actual = len(set(doc.metadata['source'] for doc in documents))
print(f'Expected PDFs: {expected}')
print(f'Succefully loaded PDFs: {actual}')
print(documents[0].metadata)

Expected PDFs: 5
Succefully loaded PDFs: 5
{'producer': 'iTextSharp 4.0.3 (based on iText 2.0.2)', 'creator': 'CommonLook Office-2.1.15.47', 'creationdate': '2026-02-03T08:15:53-05:00', 'author': 'CDRH FDA', 'keywords': '', 'moddate': '2026-02-03T09:10:12-05:00', 'nccl_app': 'Office', 'nccl_standard': 'Section 508; WCAG 2.0 AA; PDF/UA', 'nccl_status': 'Passed', 'subject': '', 'title': 'Computer Software Assurance for Production and Quality Management System Software', 'part': '1', 'source': '../data/FDA/guidance-computer-software-assurance-production-quality-system.pdf', 'total_pages': 41, 'page': 0, 'page_label': '1'}


In [3]:
# print file name and number of pages of each files
file_info = set((doc.metadata['title'], doc.metadata['total_pages']) for doc in documents)

for title, pages in file_info:
    print(f'{title}: {pages} pages')

Investigating Out-of-Specification (OOS) Test Results for Pharmaceutical Production: 17 pages
SOPP 8217: Administrative Process and Review Management of Investigational New Drug Applications: 31 pages
Computer Software Assurance for Production and Quality Management System Software: 41 pages
SOPP 8201: Administrative Processing of Clinical Holds for Investigational New Drug Applications: 15 pages
SOPP 8212: Breakthrough Therapy Products - Designation and Management: 25 pages


In [4]:
# spot-check first 500 characters in first document
print(documents[0].page_content[:500])

Contains Nonbinding Recommendations
Computer Software Assurance for 
Production and Quality Management 
System Software
Guidance for Industry and 
Food and Drug Administration Staff 
Document issued on February 3, 2026.
This document supersedes “Computer Software Assurance for Production 
and Quality System Software,” issued September 24, 2025. 
For questions about this document regarding CDRH-regulated devices, contact the Compliance 
and Quality Staff at 301-796-5577 or by email at CaseforQual


## Sprint 1 - Stage 2: Chunking
### Functions
- Split documents into chunks that can be quickly retrieved and integrated into the model prompt.
- Configure chunk_size and overlap parameters
- Inspect chunk content that each chunk size is within the chunk_size character limit

In [5]:
# import library
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

# print the first 3 chunks
for i, chunk in enumerate(chunks[:3]):
    print(f'\n------ Chunk {i+1} ------')

    # Chunk size remains within 500 characters limit
    print(f'Characters: {len(chunk.page_content)}')
    print(f'Source: {chunk.metadata['source']}')
    print(f'Page: {chunk.metadata['page']}')
    
    # inspect chunk output
    print(chunk.page_content[:500])


------ Chunk 1 ------
Characters: 438
Source: ../data/FDA/guidance-computer-software-assurance-production-quality-system.pdf
Page: 0
Contains Nonbinding Recommendations
Computer Software Assurance for 
Production and Quality Management 
System Software
Guidance for Industry and 
Food and Drug Administration Staff 
Document issued on February 3, 2026.
This document supersedes “Computer Software Assurance for Production 
and Quality System Software,” issued September 24, 2025. 
For questions about this document regarding CDRH-regulated devices, contact the Compliance

------ Chunk 2 ------
Characters: 459
Source: ../data/FDA/guidance-computer-software-assurance-production-quality-system.pdf
Page: 0
and Quality Staff at 301-796-5577 or by email at CaseforQuality@fda.hhs.gov. For questions 
about this document regarding CBER-regulated devices, contact the Office of Communication, 
Outreach, and Development (OCOD) at 800-835-4709 or 240-402-8010, or by email at 
industry.biologics@fda.hhs.

## Sprint 1 - Stage 3: Embedding and Storing
### Functions
- Initialise embedding model and vector store
- Confirm that embeddings are stored in the vector store and document count matches

In [7]:
# import libraries
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [ ]:
# initialise embedding model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = HuggingFaceEmbeddings(
    model_name=model_name
)

# create a vector store
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory='../chroma_db'
)

# verify the database is persisted to disk for future use
vector_store_check = Chroma(
    persist_directory='../chroma_db',
    embedding_function=embedding_model
)

# confirm chunk count matches 
print(f'Chunks written: {len(chunks)}')
print(f'Chunks in Chroma: {vector_store_check._collection.count()}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chunks written: 707
Chunks in Chroma: 707


## Sprint 1 - Stage 4: Similarity Search and Retrieval
### Functions
- Write query test set and run similarity search on the vector store
- Validate retrieval output that the most relevant chunks are returned

In [29]:
query = [
        'What should be included in a full-scale OOS investigation?',
         'During an OOS investigation, can the retest be done by the same analyst who performed the original test?',
         'What is the difference between process risk and medical device risk?',
         'Examples of unscripted testing.',
         'What is the timeframe for FDA to respond after receiving a complete response to a clinical hold?',
         'Who has the final authority to decide whether to issue a clinical hold on an IND?',
         'How long does CBER have to notify a sponsor whether their product has received breakthrough therapy designation?',
         'What happens if a sponsor does not respond to an Intent to Rescind Breakthrough Therapy Designation letter?',
         'When does an original IND go into effect if FDA does not notify the sponsor of a clinical hold?',
         'What is the review timeline for a breakthrough therapy designation request under this SOPP?'
         ]

for i, q in enumerate(query):
    # run similarity search and return the top 3 most relevant chunks
    result = vector_store.similarity_search(query=q,
                                        k=3)
    print(f'Query {i+1}: {q}\n')

    for doc in result:
        # inspect retrieval chunk putput
        print(f'Source: {doc.metadata["source"]}')
        print(f'\nPage number: {doc.metadata["page_label"]}\n')
        print(f'Chunk retrieved: \n{doc.page_content}')
        print('\n'+'-'*12)
    
    print('\n'+'='*30)

Query 1: What should be included in a full-scale OOS investigation?

Source: ../data/FDA/19287685_L2-OOS.pdf

Page number: 9

Chunk retrieved: 
testing results appear to be accurate, a full-scale OOS investigation using a predefined procedure 
should be conducted.  The objective of such an investigation should be to identify the root cause 
of the OOS result and take appropriate corrective action and preventive action.
7  A full-scale 
investigation should include a review of production and sampling procedures and will often 
include additional laboratory testing. Such investigations should be given the highest priority.

------------
Source: ../data/FDA/19287685_L2-OOS.pdf

Page number: 9

Chunk retrieved: 
manufacturer or at multiple manufacturing sites), all sites potentially involved should be 
included in the investigation.  Other potential problems should be identified and investigated. 
 
The records and documentation of the manufacturing process should be fully reviewed to 
det

### Test Log — Similarity Search Validation

| Query no. | Query | Source(s) | Top-3 Relevance | Notes |
|---|---|---|---|---|
| 1 | What should be included in a full-scale OOS investigation? | 19287685_L2-OOS.pdf | High | Top-2 chunks directly answer the query; top-3 is general overview, slightly less relevant. |
| 2 | Can the retest be done by the same analyst who performed the original test? | 19287685_L2-OOS.pdf | High | All three chunks relevant to retest procedure and analyst qualification. |
| 3 | What is the difference between process risk and medical device risk? | guidance-computer-software-assurance-production-quality-system.pdf | High | All three chunks directly define and contrast the two terms. |
| 4 | Examples of unscripted testing. | guidance-computer-software-assurance-production-quality-system.pdf | High | Top-1 and top-3 chunks overlap significantly in content, likely due to chunk size/overlap settings. |
| 5 | What is the timeframe for FDA to respond after receiving a complete response to a clinical hold? | SOPP-8201-Administrative-Processing-Clinical-Holds-INDs_V11.pdf | High | All three chunks relevant to clinical hold response timelines. |
| 6 | Who has the final authority to decide whether to issue a clinical hold on an IND? | SOPP-8201-Administrative-Processing-Clinical-Holds-INDs_V11.pdf | Medium | Top-1 answers the query (Clinical Division Director); top-2 is tangential (describes post-hold letter process, not decision authority). |
| 7 | How long does CBER have to notify a sponsor whether their product has received breakthrough therapy designation? | SOPP-8212-Breakthrough-Therapy-Products-Designation-and-Management-V8-b.pdf | High | All three chunks relevant; top-1 gives the direct 60-day answer. |
| 8 | What happens if a sponsor does not respond to an Intent to Rescind Breakthrough Therapy Designation letter? | SOPP-8212-Breakthrough-Therapy-Products-Designation-and-Management-V8-b.pdf | High | All three chunks relevant to the rescind process and consequences. |
| 9 | When does an original IND go into effect if FDA does not notify the sponsor of a clinical hold? | SOPP-8217 and SOPP-8201 | High | Retrieved chunks from two different SOPPs both citing the same 30-day rule (21 CFR 312.40)|
| 10 | What is the review timeline for a breakthrough therapy designation request under this SOPP? | SOPP-8212-Breakthrough-Therapy-Products-Designation-and-Management-V8-b.pdf | High | All three chunks relevant to the request review process. |

**Summary:** 
- 9/10 queries returned highly relevant top-3 chunks.
- 1 query (Query 6) returned partially relevant results. 
- No retrieval failures observed. 
- Chunk overlap noted for Query 4 as a potential tuning consideration in later sprints.
- Cross-document references (SOPP-8217 and SOPP-8201) noted for Query 9, relevant for source citation design in sprint 2.

## Sprint 2 - Stage 1: Groq API set up
### Functions
- Set up GROQ API key
- Test API calling and confirm a response is returned
- Create call_llm() function to call the LLM with prompt

In [34]:
# import libraries
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq

In [33]:
# load env file
load_dotenv()

api_key = os.getenv('GROQ_API_KEY')

# confirm API key is loaded successfully
print('API key loaded:', api_key is not None)

API key loaded: True


In [35]:
# initialise the llm
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=api_key
)

# test to invoke for a response
prompt = "What is GMP in pharmaceutical industry?"
response = llm.invoke(prompt)

print(response.content)

In the pharmaceutical industry, GMP stands for Good Manufacturing Practice. It refers to a set of guidelines and regulations that ensure the quality, safety, and efficacy of pharmaceutical products. GMP is a critical aspect of the pharmaceutical industry, as it helps to prevent contamination, mix-ups, and other errors that could affect the quality of the final product.

GMP guidelines cover various aspects of pharmaceutical manufacturing, including:

1. **Quality management**: Establishing a quality management system to ensure that products are manufactured, tested, and released in accordance with regulatory requirements.
2. **Facility and equipment**: Ensuring that manufacturing facilities and equipment are designed, constructed, and maintained to prevent contamination and ensure product quality.
3. **Personnel**: Training personnel on GMP principles, procedures, and practices to ensure that they understand their roles and responsibilities in maintaining product quality.
4. **Material

In [ ]:
# define call_llm() function to return response content when called
def call_llm(prompt):
    """Send a prompt to the LLM and return the response text."""
    response=llm.invoke(prompt)
    return response.content

# test the call_llm() function
result = call_llm("What is GMP?")
print(result)

GMP stands for Good Manufacturing Practice. It is a set of guidelines and regulations that ensure the quality, safety, and consistency of pharmaceuticals, food, and other products. The main goal of GMP is to minimize the risk of contamination, mix-ups, and other errors that could affect the quality of the final product.

GMP guidelines cover various aspects of manufacturing, including:

1. **Facilities and equipment**: The design, construction, and maintenance of facilities and equipment to prevent contamination and ensure proper functioning.
2. **Personnel**: The training, qualifications, and hygiene practices of personnel involved in manufacturing.
3. **Raw materials**: The sourcing, testing, and handling of raw materials to ensure their quality and purity.
4. **Production**: The manufacturing process, including batch production, labeling, and packaging.
5. **Quality control**: The testing and inspection of products to ensure they meet specifications and are free from contamination.
